# Phase 6 DAEAC FCBA RR Late-Fusion NSV Pretrain

This notebook pretrains the 3-class `N/S/V` FCBA + RR late-fusion model. The attached bundle must be generated from `configs/phase6_daeac_record_splits_nsv.yaml`, so all configured `.npz` files contain `rr_features` and no `F` samples.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
os.environ['PYTHONUNBUFFERED'] = '1'

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
CONFIG = 'configs/phase6_daeac_fcba_latefusion_rr_nsv.yaml'

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Kaggle working:', WORK.exists())
print('Repo path:', ECG)

## 1. Clone Repo

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    !git clone --branch {BRANCH} {REPO_URL} /kaggle/working/Best-thesis-in-council
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), f'Missing repo at {ECG}. Clone or upload it first.'
os.chdir(ECG)
!git status --short || true

## 2. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. Copy NSV RR Bundle

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one record-split bundle, found {candidate_dirs}'

DEST = Path('data/processed/phase6_daeac_record_splits_nsv')
DEST.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, DEST / src.name)
    print('copied', src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, DEST / manifest.name)

required = ('ds1_train.npz', 'ds1_val.npz', 'ds2_test.npz')
for name in required:
    path = DEST / name
    assert path.exists(), f'Missing required split file: {path}'
    with np.load(path, allow_pickle=True) as data:
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
        assert data['rr_features'].shape == (len(data['x']), 7), (name, data['x'].shape, data['rr_features'].shape)
        assert np.isfinite(data['rr_features']).all(), f'{name} rr_features contains NaN/Inf.'
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        if 'y' in data.files:
            assert set(np.unique(data['y']).tolist()).issubset({0, 1, 2}), (name, np.unique(data['y']).tolist())

!ls -lh data/processed/phase6_daeac_record_splits_nsv

## 4. Validate

In [ ]:
!python scripts/check_repo.py
subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', CONFIG], check=True)

## 5. Optional Smoke Pretrain

In [ ]:
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
        '--config', CONFIG,
        '--epochs', '1',
        '--max-source-samples', '512',
        '--max-val-samples', '512',
        '--checkpoint-prefix', 'daeac_fcba_rr_nsv_base_smoke',
    ], check=True)

## 6. Full Pretrain

In [ ]:
subprocess.run([
    sys.executable, '-u', 'scripts/phase6_daeac_paper/01_train_base.py',
    '--config', CONFIG,
], check=True)

## 7. Evaluate Best Checkpoint on DS2 Test

In [ ]:
summary_path = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv/metrics/daeac_base_train_summary.json')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
best_checkpoint = Path(summary['best_checkpoint'])
assert best_checkpoint.exists(), best_checkpoint

subprocess.run([
    sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
    '--config', CONFIG,
    '--checkpoint', str(best_checkpoint),
    '--method-name', 'daeac_fcba_rr_nsv_base_best',
    '--dataset', 'target',
], check=True)

metrics_path = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv/metrics/daeac_fcba_rr_nsv_base_best_target_test_metrics.json')
print(metrics_path.read_text())

## 8. Package Outputs

In [ ]:
!zip -r /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_pretrain_outputs.zip outputs/phase6_daeac_fcba_latefusion_rr_nsv